In [ ]:
"""
Travel Itinerary AI Backend — powered by Groq (free tier)
Uses the OpenAI-compatible Groq API with llama-3.3-70b-versatile.

Install:  pip install openai
Run:      GROQ_API_KEY="your-key" python groq_travel.py
Get key:  https://console.groq.com/keys
"""

import os
import json
import uuid
import re
from datetime import datetime
from pathlib import Path
from openai import OpenAI

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "YOUR_API_KEY_HERE")
MODEL_NAME   = "llama-3.3-70b-versatile"   # free-tier Groq model
OUTPUT_DIR   = Path("itineraries")
OUTPUT_DIR.mkdir(exist_ok=True)

# Groq via OpenAI-compatible client
client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1",
)

In [ ]:
# ── Groq helper ───────────────────────────────────────────────────────────────
def ask_groq(prompt: str) -> dict | list:
    """Send a prompt to Groq and return the parsed JSON response."""
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": "You are a travel planner. Always respond with valid JSON only. No markdown, no extra text."},
            {"role": "user",   "content": prompt},
        ],
        temperature=0.7,
        max_tokens=4096,
        response_format={"type": "json_object"},
    )
    raw = response.choices[0].message.content.strip()

    # Strip markdown fences if model wraps anyway
    raw = re.sub(r"^```(?:json)?\s*", "", raw)
    raw = re.sub(r"\s*```$", "", raw)

    parsed = json.loads(raw)
    # Unwrap {"events": [...]} if needed
    if isinstance(parsed, dict) and "events" in parsed and len(parsed) == 1:
        return parsed["events"]
    return parsed

In [ ]:
# ── Prompt builders ───────────────────────────────────────────────────────────
def build_itinerary_prompt(request: dict) -> str:
    return f"""
A user submitted this travel request:
{json.dumps(request)}

Return a single JSON object with these fields only:
{{
  "itinerary_id": "<uuid>",
  "user_id": "{request.get('user_id')}",
  "trip_name": "<short title>",
  "destination": "<city, country>",
  "origin": "<city, country>",
  "date_from": "<YYYY-MM-DD>",
  "date_to": "<YYYY-MM-DD>",
  "duration_days": <int>,
  "currency": "<ISO 4217>",
  "travel_style": "<budget|comfort|luxury>",
  "travelers": {{"adults": <int>, "children": <int>}},
  "itinerary_link": "https://trips.example.com/<itinerary_id>",
  "event_ids": [],
  "created_at": "<ISO 8601 timestamp>",
  "status": "draft"
}}
"""


def build_events_prompt(itinerary: dict, request: dict) -> str:
    itin_id = itinerary["itinerary_id"]
    itin_link = itinerary.get("itinerary_link", f"https://trips.example.com/{itin_id}")
    return f"""
Generate travel events for this itinerary (id: {itin_id}).
Trip: {itinerary.get('origin')} → {itinerary.get('destination')}, {itinerary.get('date_from')} to {itinerary.get('date_to')}.
Travelers: {json.dumps(request.get('travelers', {}))}. Style: {itinerary.get('travel_style')}.

Return a JSON object: {{"events": [...]}} where each event is MINIMAL — only the fields listed below.
Include: outbound flight, return flight, hotel stay, 2 restaurants, 2 activities.

FLIGHT:
{{
  "event_id": "<uuid>", "type": "flight", "itinerary_id": "{itin_id}",
  "tz": "<IANA timezone, e.g. America/Los_Angeles>",
  "date": "<YYYY-MM-DD>",
  "start_time": "<HH:MM>", "end_time": "<HH:MM>",
  "airline": "<name>", "flight_number": "<code>",
  "origin_airport": "<IATA>", "destination_airport": "<IATA>",
  "itinerary_link": "{itin_link}", "user_id": "{itinerary.get('user_id')}"
}}

HOTEL:
{{
  "event_id": "<uuid>", "type": "hotel", "itinerary_id": "{itin_id}",
  "tz": "<IANA timezone>",
  "start_time": "<check-in HH:MM>", "end_time": "<check-out HH:MM>",
  "check_in_date": "<YYYY-MM-DD>", "check_out_date": "<YYYY-MM-DD>",
  "hotel_name": "<name>", "room_type": "<type>",
  "confirmation_number": "<alphanumeric>",
  "itinerary_link": "{itin_link}", "user_id": "{itinerary.get('user_id')}"
}}

RESTAURANT:
{{
  "event_id": "<uuid>", "type": "restaurant", "itinerary_id": "{itin_id}",
  "tz": "<IANA timezone>",
  "date": "<YYYY-MM-DD>",
  "start_time": "<HH:MM>", "end_time": "<HH:MM>",
  "restaurant_name": "<name>", "reservation_id": "<alphanumeric>",
  "itinerary_link": "{itin_link}", "user_id": "{itinerary.get('user_id')}"
}}

ACTIVITY:
{{
  "event_id": "<uuid>", "type": "activity", "itinerary_id": "{itin_id}",
  "tz": "<IANA timezone>",
  "date": "<YYYY-MM-DD>",
  "start_time": "<HH:MM>", "end_time": "<HH:MM>",
  "activity_name": "<name>", "booking_id": "<alphanumeric or null>",
  "itinerary_link": "{itin_link}", "user_id": "{itinerary.get('user_id')}"
}}
"""

In [ ]:
# ── Core functions ────────────────────────────────────────────────────────────
def generate_itinerary(request: dict) -> dict:
    """Generate an itinerary JSON from a user request."""
    prompt    = build_itinerary_prompt(request)
    itinerary = ask_groq(prompt)

    itinerary.setdefault("itinerary_id", str(uuid.uuid4()))
    itinerary["user_id"]    = request.get("user_id", itinerary.get("user_id"))
    itinerary["created_at"] = datetime.utcnow().isoformat() + "Z"
    itinerary["event_ids"]  = []
    # Ensure itinerary_link is set
    itinerary.setdefault("itinerary_link", f"https://trips.example.com/{itinerary['itinerary_id']}")

    return itinerary


def generate_events(itinerary: dict, request: dict) -> list[dict]:
    """Generate individual event objects linked to the itinerary."""
    prompt = build_events_prompt(itinerary, request)
    events = ask_groq(prompt)
    if not isinstance(events, list):
        events = events.get("events", [])

    seen_ids = set()
    for ev in events:
        eid = ev.get("event_id", "")
        if not eid or eid in seen_ids or "uuid" in eid.lower():
            ev["event_id"] = str(uuid.uuid4())
        ev["itinerary_id"] = itinerary["itinerary_id"]
        seen_ids.add(ev["event_id"])

    return events


def save_json(data: dict | list, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    print(f"  ✔ Saved {path}")


def plan_trip(request: dict) -> dict:
    """
    Full pipeline:
      1. Generate itinerary
      2. Generate events
      3. Link event IDs back into the itinerary
      4. Save everything to disk
    Returns the final itinerary dict.
    """
    print("\n🌍 Generating itinerary …")
    itinerary = generate_itinerary(request)

    print("📅 Generating events …")
    events = generate_events(itinerary, request)

    itinerary["event_ids"] = [e["event_id"] for e in events]

    trip_dir = OUTPUT_DIR / itinerary["itinerary_id"]
    trip_dir.mkdir(parents=True, exist_ok=True)

    save_json(itinerary, trip_dir / "itinerary.json")

    events_index = []
    for ev in events:
        ev_type = ev.get("type", "event")
        date    = ev.get("date") or ev.get("check_in_date", "day00")
        ev_id   = ev["event_id"][:8]
        fname   = f"{date}_{ev_type}_{ev_id}.json"
        save_json(ev, trip_dir / fname)
        events_index.append({"event_id": ev["event_id"], "type": ev_type, "date": date, "file": fname})

    save_json(events_index, trip_dir / "events_index.json")
    save_json({"itinerary": itinerary, "events": events}, trip_dir / "full_trip.json")

    print(f"\n✅ Trip saved → {trip_dir}/")
    return itinerary

In [ ]:
# ── Example request ───────────────────────────────────────────────────────────
SAMPLE_REQUEST = {
    "user_id": "usr_8f3a21bc",
    "origin": "Los Angeles, USA",
    "destination": "Tokyo, Japan",
    "date_from": "2026-06-10",
    "date_to":   "2026-06-20",
    "travelers": {
        "adults": 2,
        "children": 0
    },
    "travel_style": "comfort",
    "currency": "USD",
    "budget_total": 6000,
    "preferences": {
        "dietary": ["vegetarian"],
        "interests": ["culture", "food", "anime", "nature"],
        "accommodation_type": "hotel",
        "avoid": ["extreme sports"]
    },
    "special_requests": "Anniversary trip. Surprise dinner on day 5."
}

In [ ]:
# ── Run ───────────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    if GROQ_API_KEY == "YOUR_API_KEY_HERE":
        print("⚠️  Set GROQ_API_KEY environment variable before running.")
        print("   export GROQ_API_KEY='your-key-here'")
        print("   Get a free key at: https://console.groq.com/keys")
    else:
        itinerary = plan_trip(SAMPLE_REQUEST)
        print(f"\n🗺  Trip: {itinerary.get('trip_name')}")
        print(f"   {itinerary.get('date_from')} → {itinerary.get('date_to')}")
        print(f"   Events linked: {len(itinerary.get('event_ids', []))}")
        print(f"   Itinerary link: {itinerary.get('itinerary_link')}")